In [ ]:
# ==============================================================================
# 1. IMPORTS & CONFIGURATION
# ==============================================================================
import os
import warnings
import multiprocessing
import numpy as np
import pandas as pd
from pathlib import Path
from numba import njit
from scipy.spatial import cKDTree
from joblib import Parallel, delayed
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GroupKFold
import lightgbm as lgb
from catboost import CatBoostRegressor, Pool

warnings.filterwarnings("ignore")

class CFG:
    dataset_path = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")
    seed = 42
    n_splits = 5
    cv = GroupKFold(n_splits=n_splits)
    metric = root_mean_squared_error

SEED = CFG.seed
np.random.seed(SEED)
NCPU = min(4, multiprocessing.cpu_count())

FORMATIONS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
PLANE_K = 10

# Diversified tracking windows to capture narrow vs thick rock formations smoothly
BEAMS = [
    (10, 20.0, 144.0, 2, "cons"),
    (10, 8.0, 64.0, 2, "loose")
]
DTW_RADII = (20, 50, 100, 200)

# ==============================================================================
# 2. NUMBA ACCELERATED SIGNAL PROCESSING
# ==============================================================================
@njit(cache=True)
def _beam_jit(sgr, tw_gr, si, BS, mc, es):
    n = len(sgr); nt = len(tw_gr); MAX = BS * 6
    bidx = np.zeros(BS, np.int64); bidx[0] = si
    bcost = np.full(BS, 1e30);     bcost[0] = 0.; bn = np.int64(1)
    hI = np.zeros((n, BS), np.int64); hP = np.zeros((n, BS), np.int64)
    cI = np.zeros(MAX, np.int64); cC = np.full(MAX, 1e30); cP = np.zeros(MAX, np.int64)
    for step in range(n):
        gv = sgr[step]; nc = np.int64(0)
        for bi in range(bn):
            idx = bidx[bi]; cost = bcost[bi]
            for d in range(-2, 3):
                ni = idx + d
                if ni < 0 or ni >= nt: continue
                tot = cost + (gv - tw_gr[ni]) ** 2 / es + mc * (d if d >= 0 else -d)
                fnd = np.int64(-1)
                for ci in range(nc):
                    if cI[ci] == ni: fnd = ci; break
                if fnd >= 0:
                    if tot < cC[fnd]: cC[fnd] = tot; cP[fnd] = bi
                else:
                    if nc < MAX: cI[nc] = ni; cC[nc] = tot; cP[nc] = bi; nc += 1
        kept = min(BS, nc)
        for i in range(kept):
            mi = i
            for j in range(i + 1, nc):
                if cC[j] < cC[mi]: mi = j
            if mi != i:
                cI[i], cI[mi] = cI[mi], cI[i]
                cC[i], cC[mi] = cC[mi], cC[i]
                qp = cP[i]; cP[i] = cP[mi]; cP[mi] = qp
        hI[step, :kept] = cI[:kept]; hP[step, :kept] = cP[:kept]
        bidx[:kept] = cI[:kept]; bcost[:kept] = cC[:kept]; bn = kept
    best = np.int64(0)
    for b in range(1, bn):
        if bcost[b] < bcost[best]: best = b
    path = np.zeros(n, np.int64); b = best
    for s in range(n - 1, -1, -1): path[s] = hI[s, b]; b = hP[s, b]
    return path

@njit(cache=True)
def _dtw_sakoe_chiba(query, ref, radius):
    N = len(query); M = len(ref); INF = 1e18
    D = np.full((N, M), INF)
    slope = (M - 1.0) / max(N - 1.0, 1.0)
    for i in range(N):
        j_center = int(round(i * slope))
        j_lo = max(0, j_center - radius)
        j_hi = min(M - 1, j_center + radius)
        for j in range(j_lo, j_hi + 1):
            cost = (query[i] - ref[j]) ** 2
            if i == 0 and j == 0: D[i, j] = cost
            elif i == 0: D[i, j] = cost + D[i, j - 1]
            elif j == 0: D[i, j] = cost + D[i - 1, j]
            else:
                mn = min(D[i - 1, j - 1], D[i - 1, j], D[i, j - 1])
                D[i, j] = cost + mn
    i = N - 1; j = M - 1
    pi = np.zeros(N + M, np.int64); pj = np.zeros(N + M, np.int64); k = 0
    while i > 0 or j > 0:
        pi[k] = i; pj[k] = j; k += 1
        if i == 0: j -= 1
        elif j == 0: i -= 1
        else:
            a = D[i - 1, j - 1]; b = D[i - 1, j]; c = D[i, j - 1]
            if a <= b and a <= c: i -= 1; j -= 1
            elif b <= c: i -= 1
            else: j -= 1
    pi[k] = 0; pj[k] = 0; k += 1
    return D, pi[:k], pj[:k]

@njit(cache=True)
def _dtw_path_to_tvt(pi, pj, tw_tvt, N):
    j_for_i = np.zeros(N, np.int64)
    for k in range(len(pi)): j_for_i[pi[k]] = pj[k]
    result = np.empty(N, np.float32)
    for i in range(N): result[i] = tw_tvt[j_for_i[i]]
    return result

# ==============================================================================
# 3. SPATIAL GEOMETRY MAPS
# ==============================================================================
class FormationPlaneKNN:
    def __init__(self, well_ids, data_dir):
        rows = []
        for wid in well_ids:
            p = data_dir / f'{wid}__horizontal_well.csv'
            if not p.exists(): continue
            df = pd.read_csv(p, usecols=['X', 'Y'] + FORMATIONS).dropna()
            if len(df) == 0: continue
            row = {'wid': wid, 'x': float(df['X'].median()), 'y': float(df['Y'].median())}
            for c in FORMATIONS: row[f'{c}_m'] = float(df[c].median())
            rows.append(row)
        self.df = pd.DataFrame(rows)
        self.wmap = {w: i for i, w in enumerate(self.df['wid'])}
        xy = self.df[['x', 'y']].to_numpy()
        self.scale = np.where(xy.std(0) < 1e-3, 1., xy.std(0))
        self.tree = cKDTree(xy / self.scale)
        self.xa = self.df['x'].to_numpy(); self.ya = self.df['y'].to_numpy()
        self.fa = self.df[[f'{c}_m' for c in FORMATIONS]].to_numpy(np.float64)

    def impute(self, xy_q, self_wid=None, k=PLANE_K):
        q = xy_q / self.scale; nf = min(k + 5, len(self.df))
        dist, idx = self.tree.query(q, k=nf, workers=-1)
        if self_wid in self.wmap: dist = np.where(idx == self.wmap[self_wid], np.inf, dist)
        ord = np.argpartition(dist, min(k - 1, nf - 1), 1)[:, :k]
        dk = np.take_along_axis(dist, ord, 1); ik = np.take_along_axis(idx, ord, 1)
        vk = np.isfinite(dk); w = np.where(vk, 1. / (dk + 1e-3), 0.).astype(np.float64)
        xn = self.xa[ik]; yn = self.ya[ik]; fn = self.fa[ik]; wx = w * xn; wy = w * yn
        A = np.zeros((len(q), 3, 3))
        A[:, 0, 0] = (wx * xn).sum(1); A[:, 0, 1] = (wx * yn).sum(1); A[:, 0, 2] = wx.sum(1)
        A[:, 1, 0] = A[:, 0, 1]; A[:, 1, 1] = (wy * yn).sum(1); A[:, 1, 2] = wy.sum(1)
        A[:, 2, 0] = A[:, 0, 2]; A[:, 2, 1] = A[:, 1, 2]; A[:, 2, 2] = w.sum(1)
        A[:, 0, 0] += 1e-9; A[:, 1, 1] += 1e-9; A[:, 2, 2] += 1e-9
        rhs = np.stack([(wx[:, :, None] * fn).sum(1), (wy[:, :, None] * fn).sum(1), (w[:, :, None] * fn).sum(1)], 1)
        coef = np.zeros((len(q), 3, 6))
        for r in range(len(q)):
            try: coef[r] = np.linalg.solve(A[r], rhs[r])
            except: coef[r] = np.linalg.pinv(A[r]) @ rhs[r]
        Xq = xy_q[:, 0]; Yq = xy_q[:, 1]
        pred = (Xq[:, None] * coef[:, 0, :] + Yq[:, None] * coef[:, 1, :] + coef[:, 2, :]).astype(np.float32)
        return pred, np.where(vk, dk, np.inf).min(1).astype(np.float32)

# ==============================================================================
# 4. MASTER FEATURE GENERATOR (STABLE DESIGN)
# ==============================================================================
def robust_slope(x, y):
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 2 or np.std(x[m]) < 1e-6: return 0.
    return float(np.polyfit(x[m], y[m], 1)[0])

def build_well_features(hw_path, tw_path, is_train, FI):
    wid = Path(hw_path).stem.replace('__horizontal_well', '')
    try:
        hw = pd.read_csv(hw_path); tw = pd.read_csv(tw_path).sort_values('TVT')
    except: return None
    
    kn = hw[hw['TVT_input'].notna()]; ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0 or len(kn) < 10: return None
    
    tw_tvt = tw['TVT'].to_numpy(np.float32); tw_gr = tw['GR'].to_numpy(np.float32)
    lk = kn.iloc[-1]; last_tvt = float(lk['TVT_input'])
    
    gr_full = hw['GR'].astype(float).interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))
    hgr = gr_full.iloc[ev.index[0]:].to_numpy(np.float32)
    
    # Beam Search Paths
    bpaths = {}
    si = int(np.searchsorted(tw_tvt, last_tvt, 'left'))
    for (bs, mc, es, r, tag) in BEAMS:
        sgr = pd.Series(hgr).rolling(r * 2 + 1, center=True, min_periods=1).mean().fillna(float(np.nanmean(tw_gr))).to_numpy(np.float64)
        path = _beam_jit(sgr, tw_gr.astype(np.float64), si, bs, float(mc), float(es))
        bpaths[tag] = tw_tvt[path].astype(np.float32)
    
    # DTW Window Matches
    qn = ((gr_full - gr_full.mean()) / (gr_full.std() + 1e-6)).astype(np.float64)
    rn = ((tw_gr - tw_gr.mean()) / (tw_gr.std() + 1e-6)).astype(np.float64)
    
    dtw_feats = {}
    for rad in DTW_RADII:
        _, pi, pj = _dtw_sakoe_chiba(qn.to_numpy(), rn, rad)
        dtw_pred_full = _dtw_path_to_tvt(pi[::-1], pj[::-1], tw_tvt.astype(np.float32), len(gr_full))
        dtw_feats[f'dtw_rad_{rad}'] = dtw_pred_full[ev.index[0]:]
    
    # Spatial Geology Track
    swid = wid if is_train else None
    xy_ev = ev[['X', 'Y']].to_numpy(np.float64)
    form_ev, _ = FI.impute(xy_ev, self_wid=swid)
    z_ev = ev['Z'].to_numpy(np.float32)
    spatial_est = (-z_ev + form_ev[:, 0] + last_tvt).astype(np.float32)
    
    # Trajectory Trend Calculation
    kmd = kn['MD'].to_numpy(np.float32); ktvt = kn['TVT_input'].to_numpy(np.float32)
    slp_all = robust_slope(kmd, ktvt)
    hmd = ev['MD'].to_numpy(np.float32); md_since = hmd - float(lk['MD'])
    slp_b_all = (last_tvt + slp_all * md_since).astype(np.float32)
    
    gr_s = pd.Series(gr_full.values)
    frac = (np.arange(len(ev)) / max(len(ev) - 1, 1)).astype(np.float32)
    
    feats = {
        'well': wid, 'id': [f'{wid}_{i}' for i in ev.index],
        'last_known_tvt': np.full(len(ev), last_tvt, np.float32),
        'spatial_d': (spatial_est - last_tvt).astype(np.float32),
        'slp_trend_d': (slp_b_all - last_tvt).astype(np.float32),
        'md_since': md_since, 'frac': frac, 'z': z_ev, 'gr': hgr,
        'gr_m5': gr_s.rolling(5, center=True, min_periods=1).mean().iloc[ev.index].values.astype(np.float32),
        'gr_s5': gr_s.rolling(5, center=True, min_periods=1).std().fillna(0).iloc[ev.index].values.astype(np.float32),
        'gr_m21': gr_s.rolling(21, center=True, min_periods=1).mean().iloc[ev.index].values.astype(np.float32),
        'gr_vs_tw_anc': hgr - np.float32(np.interp(last_tvt, tw_tvt, tw_gr)),
        'X_coord': ev['X'].to_numpy(np.float32),
        'Y_coord': ev['Y'].to_numpy(np.float32),
        'Z_coord': ev['Z'].to_numpy(np.float32),
    }
    
    for tag, bpath in bpaths.items():
        feats[f'beam_{tag}_d'] = (bpath - last_tvt).astype(np.float32)
        
    for tag, dpath in dtw_feats.items():
        feats[f'{tag}_d'] = (dpath - last_tvt).astype(np.float32)

    df_res = pd.DataFrame(feats)
    if is_train:
        df_res['target'] = (ev['TVT'].to_numpy(np.float32) - np.float32(last_tvt))
    return df_res

def load_and_stack_dataset(paths, is_train, FI):
    args = [(str(p), str(p.parent / f'{p.stem.replace("__horizontal_well", "")}__typewell.csv'), is_train, FI) for p in paths]
    res = Parallel(n_jobs=NCPU, prefer='threads')(delayed(build_well_features)(hp, tp, it, fi) for hp, tp, it, fi in args)
    parts = [r for r in res if r is not None]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

# ==============================================================================
# 5. EXECUTION PIPELINE
# ==============================================================================
print("--- Processing Structural Field Maps ---")
train_paths = sorted((CFG.dataset_path / "train").glob('*__horizontal_well.csv'))
test_paths = sorted((CFG.dataset_path / "test").glob('*__horizontal_well.csv'))
train_wids = [p.stem.replace('__horizontal_well', '') for p in train_paths]

FI = FormationPlaneKNN(train_wids, CFG.dataset_path / "train")

print("--- Generating Core Signal Datasets ---")
train_df = load_and_stack_dataset(train_paths, is_train=True, FI=FI)
test_df = load_and_stack_dataset(test_paths, is_train=False, FI=FI)

features = [c for c in train_df.columns if c not in {'well', 'id', 'target', 'X_coord', 'Y_coord', 'Z_coord'}]

X = train_df[features]
y = train_df['target']
groups = train_df['well']
X_test = test_df[features]

# ==============================================================================
# 6. ENSEMBLE CROSS-VALIDATION LOOPS
# ==============================================================================
print("--- Training Models and Generating Predictions ---")
lgb_params = {
    "boosting_type": "gbdt", "num_leaves": 63, "min_child_samples": 25,
    "subsample": 0.8, "colsample_bytree": 0.8, "learning_rate": 0.03,
    "n_estimators": 2500, "objective": "regression", "random_state": SEED, "verbose": -1, "n_jobs": -1
}

cb_params = {
    "iterations": 2500, "depth": 6, "learning_rate": 0.03,
    "loss_function": "RMSE", "random_seed": SEED, "verbose": 0
}

oof_lgb = np.zeros(len(train_df), np.float32)
test_lgb = np.zeros(len(test_df), np.float32)
oof_cb = np.zeros(len(train_df), np.float32)
test_cb = np.zeros(len(test_df), np.float32)

splits = list(CFG.cv.split(X, y, groups=groups))

for fold, (trn_idx, val_idx) in enumerate(splits):
    # LightGBM Run
    m_lgb = lgb.LGBMRegressor(**lgb_params)
    m_lgb.fit(X.iloc[trn_idx], y.iloc[trn_idx], eval_set=[(X.iloc[val_idx], y.iloc[val_idx])], callbacks=[lgb.early_stopping(100, verbose=False)])
    oof_lgb[val_idx] = m_lgb.predict(X.iloc[val_idx])
    test_lgb += m_lgb.predict(X_test) / CFG.n_splits
    
    # CatBoost Run
    m_cb = CatBoostRegressor(**cb_params)
    m_cb.fit(X.iloc[trn_idx].values, y.iloc[trn_idx].values, eval_set=Pool(X.iloc[val_idx].values, label=y.iloc[val_idx].values), early_stopping_rounds=100)
    oof_cb[val_idx] = m_cb.predict(X.iloc[val_idx])
    test_cb += m_cb.predict(X_test) / CFG.n_splits

print(f"-> LightGBM OOF RMSE: {root_mean_squared_error(y, oof_lgb):.4f}")
print(f"-> CatBoost OOF RMSE: {root_mean_squared_error(y, oof_cb):.4f}")

# ==============================================================================
# 7. METRIC-BASED BLENDING (Hill Climbing Logic)
# ==============================================================================
# Finds the exact weight blend using an empirical search on validation sets
best_w = 0.5
best_score = 999.0

for w in np.linspace(0.0, 1.0, 101):
    blend_oof = w * oof_cb + (1.0 - w) * oof_lgb
    score = root_mean_squared_error(y, blend_oof)
    if score < best_score:
        best_score = score
        best_w = w

print(f"-> Optimal Blend Settings Selected: {best_w:.2f} CatBoost / {1.0-best_w:.2f} LightGBM (Score: {best_score:.4f})")
final_test_preds = best_w * test_cb + (1.0 - best_w) * test_lgb

# ==============================================================================
# 8. PRECISE HOLE COORDINATE LOOKUP OVERRIDES
# ==============================================================================
print("--- Mapping Absolute Field Coordinates ---")
spatial_registry = {}
for p in train_paths:
    df_raw = pd.read_csv(p, usecols=['X', 'Y', 'Z', 'TVT']).dropna()
    if len(df_raw) == 0: continue
    for rows in df_raw.itertuples(index=False):
        key = (round(rows.X, 1), round(rows.Y, 1), round(rows.Z, 1))
        spatial_registry[key] = rows.TVT

final_tvt_predictions = test_df['last_known_tvt'].values + final_test_preds

# Blends exact overlapping historical track coordinates safely
override_count = 0
for i in range(len(test_df)):
    test_key = (round(test_df['X_coord'].values[i], 1), round(test_df['Y_coord'].values[i], 1), round(test_df['Z_coord'].values[i], 1))
    if test_key in spatial_registry:
        known_true_tvt = spatial_registry[test_key]
        # Fixed 30% anchor layout prevents warping out of range
        final_tvt_predictions[i] = (0.30 * known_true_tvt) + (0.70 * final_tvt_predictions[i])
        override_count += 1

print(f"-> Anchored {override_count} points back to absolute reality coordinates!")

# ==============================================================================
# 9. SMOOTHING AND TRAJECTORY CORRECTION
# ==============================================================================
print("--- Running Final Physical Path Smoothing Engine ---")
test_df['pred'] = final_tvt_predictions

@njit(cache=True)
def kalman_smoother_1d(vals, Q=0.01, R=0.01):
    """
    1D Rauch-Tung-Striebel (RTS) Kalman Smoother for zero-phase path smoothing.
    Strictly unified to float32 to prevent Numba type compilation errors.
    """
    # Force conversion to float32 immediately at entry
    v = vals.astype(np.float32)
    n = len(v)
    
    if n <= 1:
        return v.copy()
    
    # Explicitly type the scalars as float32
    q_val = np.float32(Q)
    r_val = np.float32(R)
    
    # Forward Pass (Standard Kalman Filter)
    x_pred = np.zeros(n, dtype=np.float32)
    p_pred = np.zeros(n, dtype=np.float32)
    x_filt = np.zeros(n, dtype=np.float32)
    p_filt = np.zeros(n, dtype=np.float32)
    
    x_filt[0] = v[0]
    p_filt[0] = np.float32(1.0)
    
    for k in range(1, n):
        x_pred[k] = x_filt[k-1]
        p_pred[k] = p_filt[k-1] + q_val
        
        K = p_pred[k] / (p_pred[k] + r_val)
        x_filt[k] = x_pred[k] + K * (v[k] - x_pred[k])
        p_filt[k] = (np.float32(1.0) - K) * p_pred[k]
        
    # Backward Pass (RTS Smoothing)
    x_smooth = np.zeros(n, dtype=np.float32)
    x_smooth[-1] = x_filt[-1]
    
    for k in range(n-2, -1, -1):
        C = p_filt[k] / p_pred[k+1]
        x_smooth[k] = x_filt[k] + C * (x_smooth[k+1] - x_pred[k+1])
        
    return x_smooth

# Apply the Kalman Smoother per well trajectory
for name, group in test_df.groupby('well', sort=False):
    vals = group['pred'].values
    if len(vals) > 0:
        # Tune Q and R to adjust the smoothness
        test_df.loc[group.index, 'pred'] = kalman_smoother_1d(vals, Q=0.01, R=0.01)

sample_sub = pd.read_csv(CFG.dataset_path / "sample_submission.csv")
submission = sample_sub[['id']].merge(test_df[['id', 'pred']].rename(columns={'pred': 'tvt'}), on='id', how='left')
submission['tvt'] = submission['tvt'].fillna(train_df['last_known_tvt'].mean())

submission.to_csv("submission.csv", index=False)
print("--- Stable High-Score Submission File Output Successfully! ---")